# 대만 월매출 기반 한국/미국 기업 매출 예측 분석

**목적**: 매월 10일경 선행 발표되는 대만 기업 월별 매출을 이용해, 상관계수가 높은 한국/미국 기업의 분기 매출을 1~2분기 먼저 예측한다.

**사전 준비**
1. `tw_revenue_V4.py` 로 대만 실적 수집 + 예측 실행 (`revenue.db` 생성)
2. 한국: `dataguide_fs_loader` 로 `korea_fs_data_from_DG` 적재
3. 미국: `US_FMP_FS_1_RUN_UPDATE.py` 로 `US_IS_from_FMP` 적재

**분석 흐름**
```
[1] 경로/DB 설정  →  [2] 대만 월별 전처리 (wide/MoM/YoY)
→ [3] 분기 groupby + 분기 YoY  →  [4] 한국/미국 분기 매출 로드
→ [5] 상관계수 상위 n개 추출  →  [6] 업종 검토 후 ticker 확정
→ [7] 회귀/ML 로 1~2분기 매출 예측 + 백테스트
```

In [1]:
# [1] 경로 통일 (노트북/데스크탑 공용) — tw_revenue_V4 방식
from analysis_config import setup_universal_paths, find_revenue_db, make_engine
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)

paths = setup_universal_paths()      # DATA 폴더 자동 탐색 → sys.path 등록
db_path = find_revenue_db()          # revenue.db 자동 탐색
engine = make_engine()               # MariaDB/MySQL (한국/미국 DB)

경로 설정 완료
프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
DATA 폴더:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
[경로] revenue.db: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\ODP\Taiwan_Company\revenue.db


## 2. 대만 월별 매출 전처리
50개사 월별 매출을 **날짜 index × 종목코드 columns** 의 DataFrame으로 만들고,
**월별 실적 변화(MoM)** 와 **YoY 변화** 2개의 DataFrame을 생성한다.

In [2]:
import tw_data

tw_conn = tw_data.get_tw_conn(db_path)
COMPANIES = tw_data.load_tw_companies(tw_conn)
print(f"대만 기업 수: {len(COMPANIES)}")

tw_long   = tw_data.load_tw_monthly_long(tw_conn)
tw_wide   = tw_data.monthly_wide(tw_long)     # ① 월별 매출 (date × company)
tw_mom    = tw_data.monthly_mom(tw_wide)      # ② 월별 실적 변화 (MoM %)
tw_yoy_m  = tw_data.monthly_yoy(tw_wide)      # ③ 월별 YoY 변화 (%)

tw_wide.tail(3)

대만 기업 수: 50


company_id,1216,1301,1303,2059,2301,2303,2308,2317,2327,2330,...,3665,3711,4904,4958,5880,6505,6515,6669,7769,8046
date,,,,,,,,,,,,,,,,,,,,,
2026-05-01,"59,180,335.00","14,951,422.00","28,830,982.00","3,791,303.00","17,353,659.00","22,943,755.00","58,961,817.00","859,409,333.00","15,058,220.00","416,975,163.00",...,NaN,"63,033,313.00","9,501,988.00",NaN,"6,305,060.00","57,777,216.00","1,073,481.00","84,050,473.00","4,129,556.00","4,440,287.00"
2026-06-01,"58,301,112.00","13,976,274.00","27,134,592.00","4,442,581.00","18,655,552.00","23,124,962.00","65,602,611.00","821,762,936.00","15,359,009.00","442,679,969.00",...,NaN,"65,783,102.00","9,622,202.00",NaN,"6,519,116.00","63,683,339.00","1,460,220.00","111,371,123.00","5,301,564.00","4,684,331.00"
2026-07-01,"62,168,637.00","15,293,925.00","30,572,583.00","6,407,256.00","19,005,536.00","23,844,045.00","67,073,192.00","946,512,543.00","16,131,188.00","467,580,548.00",...,"9,630,165.00","73,783,701.00","9,252,373.00","17,600,546.00",NaN,"74,820,787.00","1,600,351.00","117,685,530.00","5,245,898.00","5,439,751.00"


In [3]:
tw_wide.tail(3)

company_id,1216,1301,1303,2059,2301,2303,2308,2317,2327,2330,...,3665,3711,4904,4958,5880,6505,6515,6669,7769,8046
date,,,,,,,,,,,,,,,,,,,,,
2026-05-01,"59,180,335.00","14,951,422.00","28,830,982.00","3,791,303.00","17,353,659.00","22,943,755.00","58,961,817.00","859,409,333.00","15,058,220.00","416,975,163.00",...,NaN,"63,033,313.00","9,501,988.00",NaN,"6,305,060.00","57,777,216.00","1,073,481.00","84,050,473.00","4,129,556.00","4,440,287.00"
2026-06-01,"58,301,112.00","13,976,274.00","27,134,592.00","4,442,581.00","18,655,552.00","23,124,962.00","65,602,611.00","821,762,936.00","15,359,009.00","442,679,969.00",...,NaN,"65,783,102.00","9,622,202.00",NaN,"6,519,116.00","63,683,339.00","1,460,220.00","111,371,123.00","5,301,564.00","4,684,331.00"
2026-07-01,"62,168,637.00","15,293,925.00","30,572,583.00","6,407,256.00","19,005,536.00","23,844,045.00","67,073,192.00","946,512,543.00","16,131,188.00","467,580,548.00",...,"9,630,165.00","73,783,701.00","9,252,373.00","17,600,546.00",NaN,"74,820,787.00","1,600,351.00","117,685,530.00","5,245,898.00","5,439,751.00"


In [5]:
tw_yoy_m.tail(3)
# tw_yoy_m.to_excel(r'C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\Data_Asnal\Taiwan_company_revenue_yoy_growth.xlsx')
tw_wide.to_excel(r'C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\대만기업_월별매출분석\Taiwan_company_revenue_202607.xlsx')

## 3. 분기(3개월) groupby → 분기 YoY growth
상관계수 측정의 기준 데이터. 한국/미국은 분기 공시이므로 대만 월별 데이터를 캘린더 분기로 합산해 비교한다.

In [15]:
tw_q_rev = tw_data.quarterly_revenue(tw_wide)   # ④ 분기 매출 (3개월 완성 분기만)
tw_q_yoy = tw_data.quarterly_yoy(tw_q_rev)       #    분기 YoY growth (%)

# 예측치(forecast 테이블, ensemble)로 연장한 버전 — 미래 분기 예측용
tw_q_rev_ext, tw_q_yoy_ext, tw_is_fc = tw_data.tw_quarterly_extended(tw_conn, model="ensemble")
print("실적 분기:", tw_q_yoy.index.min(), "→", tw_q_yoy.index.max())
print("예측 연장:", tw_q_yoy_ext.index.max(), "| 예측 포함 분기:", list(tw_q_yoy_ext.index[tw_is_fc.values].astype(str)))
tw_q_yoy.tail(4)

실적 분기: 2019Q1 → 2026Q2
예측 연장: 2027Q4 | 예측 포함 분기: ['2026Q3', '2026Q4', '2027Q1', '2027Q2', '2027Q3', '2027Q4']


company_id,1216,1301,1303,2059,2301,2303,2308,2317,2327,2330,...,3443,3653,3711,4904,5880,6505,6515,6669,7769,8046
date,,,,,,,,,,,,,,,,,,,,,
2025Q3,-1.45,-17.38,-3.38,69.55,22.09,-2.25,33.97,10.99,4.25,30.31,...,30.28,41.09,5.29,6.69,3.11,-5.58,-6.55,172.77,129.31,19.32
2025Q4,-0.08,-18.33,-5.43,59.74,15.83,2.36,41.51,22.07,19.87,20.45,...,105.88,19.29,9.65,12.06,26.87,-3.37,45.13,152.95,87.00,41.86
2026Q1,3.40,-11.05,4.64,37.83,19.20,5.49,34.00,29.68,22.70,35.13,...,62.99,11.63,17.22,6.77,-8.01,-6.51,29.74,62.03,81.10,32.15
2026Q2,3.18,4.11,27.29,156.10,30.38,16.98,47.75,39.83,35.66,36.05,...,127.64,40.40,26.74,13.63,3.94,25.72,131.44,26.01,100.64,41.66


## 4. 한국/미국 분기 매출 로드
- 한국: `korea_fs_data_from_DG`, 매출액 `item_code='M000904001'` (단위: 천원, **단일 분기 값 확정** → `cumulative=False` 기본값 그대로)
- 미국: `US_IS_from_FMP`, `item='revenue'`, 분기 행(Q1~Q4)만 (단위: USD)

> 처음 실행 시 아래 확인 셀로 item 명칭이 맞는지 점검하세요.

In [16]:
# (선택) 매출 항목 코드/명칭 확인
import kr_us_data
# kr_us_data.list_kr_revenue_items(engine).head(10)
# kr_us_data.list_us_items(engine, keyword="revenue").head(10)

In [17]:
kr_q, kr_names = kr_us_data.load_kr_quarterly_revenue(engine, start="2018-01-01")
print("한국:", kr_q.shape, "| 기간:", kr_q.index.min(), "→", kr_q.index.max())

us_q, us_names = kr_us_data.load_us_quarterly_revenue(engine, start="2018-01-01")
print("미국:", us_q.shape, "| 기간:", us_q.index.min(), "→", us_q.index.max())

한국: (33, 1582) | 기간: 2018Q1 → 2026Q1
미국: (34, 1993) | 기간: 2018Q1 → 2026Q2


In [18]:
import correlation
kr_yoy = correlation.yoy_from_quarterly(kr_q)
us_yoy = correlation.yoy_from_quarterly(us_q)

## 5. 대만 특정 기업 → 상관계수 상위 n개 한국/미국 기업 추출
`TW_CID` 를 바꿔가며 확인. `max_lag=1` 은 '대만이 1분기 선행'하는 관계까지 함께 탐색해 최적 lag을 보고한다.

> **미국 기업**은 비표준 회계분기·발표 후행성 때문에 결과 lag이 1로 잡히는 경우가 많다 — 정상이며, 6장 예측에서도 미국은 lag=1을 기본 적용한다.

In [37]:
TW_CID = "2317"   # ← 대만 종목코드 입력 (예: 2330=TSMC, 2454=MediaTek)
TOP_N  = 20

# ★ 한국/미국 별도 DataFrame 으로 분리 추출
top_kr, top_us = correlation.top_correlated_split(
    TW_CID, tw_q_yoy, kr_yoy, us_yoy,
    kr_names=kr_names, us_names=us_names,
    n=TOP_N,
    min_corr=None,     # 0.75 로 지정하면 그 이상만 표시
    min_overlap=16,    # 최소 겹침 분기 수 (허위 상관 방지: 16~20 권장)
    max_lag=1,
)
print(f"[한국] {COMPANIES.get(TW_CID, TW_CID)} ({TW_CID}) 상관 상위")
# top_kr.to_excel(r'C:\Users\82108\OneDrive\유튜브\barrons_WSJ_pw\Taiwan_revenue_analysis\foxconn.xlsx')

[한국] Hon Hai (Foxconn) (2317) 상관 상위


In [21]:
print(f"[미국] {COMPANIES.get(TW_CID, TW_CID)} ({TW_CID}) 상관 상위")
top_us

[미국] Nanya Technology (2408) 상관 상위


,ticker,name,corr,lag,n_obs
0,VIVO,VIVO,0.38,0,16
1,HTA,HTA,0.32,0,18
2,CLI,CLI,0.31,0,17
3,LHCG,LHCG,0.30,1,16
4,CO,CO,0.29,0,18
5,AIMC,AIMC,0.27,0,16
6,SVA,SVA,0.26,1,17
7,JCOM,JCOM,0.24,0,17
8,FFHL,FFHL,0.19,1,19
9,COG,COG,0.19,1,17


## 5-2. 두 기업 페어 상관분석 (근거 데이터 포함)
대만/한국/미국 어느 시장이든 **ticker 2개**를 입력하면 시장을 자동 판별해
매출 상관계수와 그 계산에 사용된 **근거 데이터**(분기 매출·YoY·사용 여부)를 반환한다.
- `used=True` 인 분기들만 상관계수 계산에 포함됨
- `max_lag=1` 지정 시 lag 0/1 중 |corr| 최대인 매핑을 자동 선택

In [11]:
# 페어 분석용 데이터 묶음 (한 번만 구성)
FRAMES = {
    "TW": (tw_q_rev, tw_q_yoy),
    "KR": (kr_q, kr_yoy),
    "US": (us_q, us_yoy),
}

In [30]:
# ★ ticker 2개 입력 — 시장 혼합 가능 (대만 vs 한국, 한국 vs 미국 등)
T1 = "2327"      # 예: 대만 TSMC
T2 = "A009150"   # 예: 한국 삼성전자 (미국이면 "NVDA" 등)

pair_summary, pair_detail = correlation.pair_correlation(
    T1, T2, FRAMES,
    max_lag=1,        # 0/1분기 선행 중 자동 선택 (고정하려면 lag=0, max_lag=0)
    min_overlap=24,
)
pair_summary

,ticker1,market1,ticker2,market2,corr,lag,n_obs,first_q,last_q
0,2327,TW,A009150,KR,0.39,1,24,2020Q2,2026Q1


In [ ]:
# 근거 데이터: 분기별 매출 원값 + YoY + 상관계산 포함 여부(used)
pair_detail

In [ ]:
# (선택) 근거 데이터 시각화 — YoY 산점도로 상관관계 눈으로 확인
u = pair_detail[pair_detail["used"]]
ycols = [c for c in pair_detail.columns if c.startswith("yoy_")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(u.index.to_timestamp(), u[ycols[0]], "-o", ms=4, label=ycols[0])
axes[0].plot(u.index.to_timestamp(), u[ycols[1]], "-s", ms=4, label=ycols[1])
axes[0].axhline(0, color="k", lw=0.6)
axes[0].set_title("Quarterly YoY (%)"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].scatter(u[ycols[0]], u[ycols[1]], s=30)
axes[1].set_xlabel(ycols[0]); axes[1].set_ylabel(ycols[1])
axes[1].set_title(f"corr = {pair_summary['corr'].iloc[0]} (n={pair_summary['n_obs'].iloc[0]})")
axes[1].grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
# 전체 매트릭스가 필요하면 (대만 전 기업 × 한국 전 기업) — 다소 시간 소요
# corr_kr_all = correlation.corr_table(tw_q_yoy, kr_yoy, "KR", kr_names, max_lag=1)
# corr_us_all = correlation.corr_table(tw_q_yoy, us_yoy, "US", us_names, max_lag=1)
# corr_kr_all.head(20)

## 6. 예측 대상/설명변수 확정
위 결과에서 **상관계수 ≥ 0.75** 이면서 **업종 관련성이 있는** 대만 기업만 남겨 ticker 리스트를 확정한다 (업종 판단은 수동).

In [ ]:
# ★ 사용자 입력 구간 ★
TARGET_MARKET = "KR"          # "KR" 또는 "US"
TARGET_TICKER = "A005930"     # 예측할 기업 (한국: A+6자리 / 미국: 티커)
TW_TICKERS    = ["2330", "2454"]   # 업종 검토 후 확정한 대만 종목코드 리스트
HORIZON       = 2             # 예측 분기 수 (1~2)
MODEL         = "ridge"       # ols / ridge / lasso / rf

# lag: 대만 t-lag 분기 YoY → 대상 t분기 YoY 매핑
#  - 미국은 비표준 회계분기 + 발표 후행성을 고려해 Q-1(lag=1)을 기본으로 반영
#  - 한국은 동분기(lag=0) 기본. 5장의 최적 lag 결과를 보고 조정 가능
LAG = 1 if TARGET_MARKET == "US" else 0

target_rev = (kr_q if TARGET_MARKET == "KR" else us_q)[TARGET_TICKER].dropna()
target_name = (kr_names if TARGET_MARKET == "KR" else us_names).get(TARGET_TICKER, TARGET_TICKER)
print(f"예측 대상: {target_name} ({TARGET_TICKER}) | 실적 마지막 분기: {target_rev.index.max()} | lag={LAG}")

## 7. 매출 예측 + 백테스트
대만 분기 YoY(실적+`tw_revenue` 예측치)를 X로, 대상 기업 분기 YoY를 y로 회귀 학습.
예측 YoY × 전년 동분기 실제 매출 = 예측 매출 금액.

In [ ]:
import predictor

pred = predictor.predict_revenue(
    target_rev, tw_q_yoy_ext, TW_TICKERS,
    tw_is_forecast=tw_is_fc, horizon=HORIZON, model=MODEL, lag=LAG,
)
pred

In [ ]:
# 백테스트: 최근 6개 분기를 확장 윈도우로 1분기씩 예측 → 정확도 확인
bt = predictor.backtest(target_rev, tw_q_yoy_ext, TW_TICKERS,
                        model=MODEL, lag=LAG, n_test=6)
bt

In [ ]:
# 모델 비교 (동일 조건에서 어느 모델이 나은지)
for m in ["ols", "ridge", "lasso", "rf"]:
    try:
        predictor.backtest(target_rev, tw_q_yoy_ext, TW_TICKERS, model=m, lag=LAG, n_test=6)
    except Exception as e:
        print(f"[{m}] 실패: {e}")

## 8. 시각화: 실적 vs 예측

In [ ]:
target_yoy = (target_rev.pct_change(4) * 100).dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# (상) 분기 매출 + 예측
ax = axes[0]
x_act = target_rev.index.to_timestamp()
ax.plot(x_act, target_rev.values, "-o", ms=3, color="#1f77b4", label="Actual")
if not pred.empty:
    px = pd.PeriodIndex(pred["quarter"], freq="Q").to_timestamp()
    ax.plot([x_act[-1], *px], [target_rev.iloc[-1], *pred["pred_revenue"]],
            "--s", color="#d62728", label=f"Predicted ({MODEL})")
ax.set_title(f"{target_name} ({TARGET_TICKER}) Quarterly Revenue & Prediction")
ax.legend(); ax.grid(alpha=0.3)

# (하) YoY: 대상 기업 vs 대만 설명변수
ax = axes[1]
ax.plot(target_yoy.index.to_timestamp(), target_yoy.values, "-o", ms=3,
        color="#1f77b4", lw=2, label=f"{TARGET_TICKER} YoY")
for t in TW_TICKERS:
    s = tw_q_yoy_ext[t].dropna()
    ax.plot(s.index.to_timestamp(), s.values, "--", alpha=0.7,
            label=f"TW {COMPANIES.get(t, t)}")
if not pred.empty:
    ax.plot(px, pred["pred_yoy_pct"], "s", color="#d62728", ms=7, label="Pred YoY")
ax.axhline(0, color="k", lw=0.6)
ax.set_title("Quarterly YoY Growth (%) — Target vs Taiwan drivers")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

In [ ]:
tw_conn.close()
engine.dispose()